In [1]:
%load_ext autoreload
%autoreload 2

import os, sys

sys.path.insert(0, "..")
from spec.enums import MainTableColumns as Cols, EventType

import pandas as pd

In [2]:
base_dir = '../../sample_data/edwards/raw/2021/'
# base_dir = '../../sample_data/edwards/raw/2019/'

In [3]:
keystrokes = pd.read_csv(os.path.join(base_dir, 'keystrokes.csv'))
keystrokes.head()

,EventID,SubjectID,AssignmentID,CodeStateSection,EventType,SourceLocation,EditType,InsertText,DeleteText,X-Metadata,ClientTimestamp,ToolInstances,CodeStateID,X-Compilable
0,0,Student1,Assign10,task1.py,File.Edit,0.0,Insert,# @@@@@@@@@@@@@\n# CS1400 - MW1\n# Assignment ??,NaN,NaN,1636696834375,PC;PP 1.1.10,NaN,1
1,1,Student1,Assign10,task1.py,File.Edit,46.0,Insert,\n,NaN,NaN,1636696835921,PC;PP 1.1.10,NaN,1
2,2,Student1,Assign10,task1.py,X-Action,NaN,NaN,NaN,NaN,Enter,1636696838793,PC;PP 1.1.10,NaN,1
3,3,Student1,Assign10,task1.py,File.Edit,47.0,Insert,\n,NaN,NaN,1636696839138,PC;PP 1.1.10,NaN,1
4,4,Student1,Assign10,task1.py,X-Action,NaN,NaN,NaN,NaN,Paste,1636696840014,PC;PP 1.1.10,NaN,1


In [4]:
keystrokes.EventType.value_counts()

EventType
File.Edit         971220
X-Keystroke       618843
X-Action          418541
Run.Program        54561
X-ReplayAction     16247
X-Paste             3856
X-Copy              3256
X-Attention         2342
Name: count, dtype: int64

In [5]:
keystrokes.EventType.value_counts()

EventType
File.Edit         971220
X-Keystroke       618843
X-Action          418541
Run.Program        54561
X-ReplayAction     16247
X-Paste             3856
X-Copy              3256
X-Attention         2342
Name: count, dtype: int64

In [6]:
# read the first 100 lines of os.path.join(base_dir, 'runs1.csv') as a text file
# it's a large file, so we will read only the first 100 lines to memory
with open(os.path.join(base_dir, 'runs1.csv'), 'r') as f:
    for i in range(100):
        print(f.readline().strip())

SubjectID,AssignmentID,File,Action,RunID,NumOutputLines,Source,OutputDestination,Output,ClientTimestamp
Student1,Assign6,task1.py,r,5,8.0,PythonRunner,,,1634536696349.0
Student1,Assign6,,o,5,,,stdout,"Fluky Number : 155
",1634536696350.0
Student1,Assign6,,o,5,,,stdout,"Fluky Number : 168
",1634536696350.0
Student1,Assign6,,o,5,,,stdout,"Fluky Number : 357
",1634536696350.0
Student1,Assign6,,o,5,,,stdout,"Fluky Number : 580
",1634536696350.0
Student1,Assign6,,o,5,,,stdout,"Fluky Number : 1895
",1634536696405.0
Student1,Assign6,,o,5,,,stdout,"Total Time : 3.58 seconds
",1634536699842.0
Student1,Assign6,,o,5,,,stdout,"Total loops : 24990004
",1634536699842.0
Student1,Assign6,,o,5,,,system,"
Process finished with exit code 0
",1634536699851.0
Student1,Assign6,task2.py,r,5,8.0,PythonRunner,,,1634622332557.0
Student1,Assign6,,o,5,,,stdout,Press Enter to continue with the simulation.,1634622332566.0
Student1,Assign6,,o,5,,,stdout,"The zookeeper saw an elephant 3.05% of nights
",1634622339140.

In [7]:
runs1 = pd.read_csv(os.path.join(base_dir, 'runs1.csv'))
runs2 = pd.read_csv(os.path.join(base_dir, 'runs2.csv'))

C:\Users\twprice\AppData\Local\Temp\ipykernel_19012\1278313186.py:1: DtypeWarning: Columns (2,6) have mixed types. Specify dtype option on import or set low_memory=False.
  runs1 = pd.read_csv(os.path.join(base_dir, 'runs1.csv'))
C:\Users\twprice\AppData\Local\Temp\ipykernel_19012\1278313186.py:2: DtypeWarning: Columns (2,6,8) have mixed types. Specify dtype option on import or set low_memory=False.
  runs2 = pd.read_csv(os.path.join(base_dir, 'runs2.csv'))


In [8]:
all_runs = pd.concat([runs1, runs2], ignore_index=True)

In [9]:
all_runs.head(20)

,SubjectID,AssignmentID,File,Action,RunID,NumOutputLines,Source,OutputDestination,Output,ClientTimestamp
0,Student1,Assign6,task1.py,r,5,8.0,PythonRunner,NaN,NaN,1.634537e+12
1,Student1,Assign6,NaN,o,5,NaN,NaN,stdout,Fluky Number : 155\n,1.634537e+12
2,Student1,Assign6,NaN,o,5,NaN,NaN,stdout,Fluky Number : 168\n,1.634537e+12
3,Student1,Assign6,NaN,o,5,NaN,NaN,stdout,Fluky Number : 357\n,1.634537e+12
4,Student1,Assign6,NaN,o,5,NaN,NaN,stdout,Fluky Number : 580\n,1.634537e+12
5,Student1,Assign6,NaN,o,5,NaN,NaN,stdout,Fluky Number : 1895\n,1.634537e+12
6,Student1,Assign6,NaN,o,5,NaN,NaN,stdout,Total Time : 3.58 seconds\n,1.634537e+12
7,Student1,Assign6,NaN,o,5,NaN,NaN,stdout,Total loops : 24990004\n,1.634537e+12
8,Student1,Assign6,NaN,o,5,NaN,NaN,system,\nProcess finished with exit code 0\n,1.634537e+12
9,Student1,Assign6,task2.py,r,5,8.0,PythonRunner,NaN,NaN,1.634622e+12


In [10]:
current_run = None
rows = []
stderr = pd.NA

ignorable_errors = [
    # Not an error: program manually stopped
    'KeyboardInterrupt',
    # Not an error: turtle drawing inturrupted manually
    'turtle.Terminator',
]
no_colon_errors = [
    'random.seed(sum)',
    'numberSpacing = (2 * number) - 1 (- len(numberInRow))',
    'seed(var2)',
]

subset = all_runs[(all_runs.Action == 'r') | (all_runs.OutputDestination == 'stderr')]
for i, row in subset.iterrows():
    # print(row.Action, row.Action == 'r')
    if row.Action == 'r':
        if current_run is not None:
            error_type = pd.NA
            if pd.notna(stderr):
                last_line = stderr.strip().split('\n')[-1].strip()
                colon_index = last_line.find(":")
                if last_line in no_colon_errors:
                    error_type = last_line.strip()
                elif colon_index != -1:
                    error_type = last_line[:colon_index].strip()
                else:
                    if last_line not in ignorable_errors:
                        print(f"Unrecognized error type: {last_line}")
                        print(f"Full error message: {stderr}")
                    error_type = pd.NA
            current_run[Cols.CompileMessageType] = error_type
            current_run[Cols.CompileMessageData] = stderr
            rows.append(current_run)
        current_run = row.to_dict()
        stderr = pd.NA
        continue

    if row.OutputDestination == 'stderr':
        output = row.Output.strip()
        if len(output) > 0:
            if pd.isna(stderr):
                stderr = ''
            stderr += output
            if row.Output.endswith('\n'):
                stderr += '\n'

runs = pd.DataFrame(rows)
runs.rename(columns={
    'File': Cols.CodeStateSection
}, inplace=True)
runs[Cols.EventType] = 'Run.Program'
runs["X-Metadata"] = 'Start'

In [11]:
runs[~pd.isna(runs.CompileMessageData)].CompileMessageType.isna().mean()

np.float64(0.3177516816782579)

In [12]:
runs[~pd.isna(runs.CompileMessageData)]

,SubjectID,AssignmentID,CodeStateSection,Action,RunID,NumOutputLines,Source,OutputDestination,Output,ClientTimestamp,CompileMessageType,CompileMessageData,EventType,X-Metadata
11,Student1,Assign6,task2.py,r,10,31.0,PythonRunner,NaN,NaN,1.634623e+12,<NA>,"Traceback (most recent call last):\nFile ""task...",Run.Program,Start
31,Student1,Assign6,flukytester.py,r,49,15.0,PythonRunner,NaN,NaN,1.634571e+12,<NA>,"Traceback (most recent call last):\nFile ""fluk...",Run.Program,Start
32,Student1,Assign6,flukytester.py,r,50,35.0,PythonRunner,NaN,NaN,1.634571e+12,<NA>,"Traceback (most recent call last):\nFile ""fluk...",Run.Program,Start
33,Student1,Assign6,flukytester.py,r,51,15.0,PythonRunner,NaN,NaN,1.634571e+12,<NA>,"Traceback (most recent call last):\nFile ""fluk...",Run.Program,Start
39,Student1,Assign6,task2.py,r,4,5.0,PythonRunner,NaN,NaN,1.634622e+12,TypeError,"Traceback (most recent call last):\nFile ""task...",Run.Program,Start
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24185,Student9,Assign13,task2.py,r,83,14.0,PythonRunner,NaN,NaN,1.639125e+12,UnboundLocalError,"Traceback (most recent call last):\nFile ""task...",Run.Program,Start
24186,Student9,Assign13,task2.py,r,84,14.0,PythonRunner,NaN,NaN,1.639125e+12,UnboundLocalError,"Traceback (most recent call last):\nFile ""task...",Run.Program,Start
24191,Student9,Assign13,task1.py,r,92,10.0,PythonRunner,NaN,NaN,1.639127e+12,ValueError,"Traceback (most recent call last):\nFile ""task...",Run.Program,Start
24192,Student9,Assign13,task1.py,r,93,11.0,PythonRunner,NaN,NaN,1.639127e+12,TypeError,"Traceback (most recent call last):\nFile ""task...",Run.Program,Start


In [13]:
runs[[Cols.SubjectID, Cols.AssignmentID, Cols.ClientTimestamp, Cols.CodeStateSection]].duplicated().mean()

np.float64(0.0)

In [14]:
merge_cols = [Cols.SubjectID, Cols.AssignmentID, Cols.ClientTimestamp, Cols.CodeStateSection, Cols.EventType, "X-Metadata"]
data_cols = [Cols.CompileMessageType, Cols.CompileMessageData]

error_events = keystrokes[keystrokes.EventType == "Run.Program"].merge(
    runs[merge_cols + data_cols],
    on=merge_cols,
    how='left',
)

In [15]:
error_events

,EventID,SubjectID,AssignmentID,CodeStateSection,EventType,SourceLocation,EditType,InsertText,DeleteText,X-Metadata,ClientTimestamp,ToolInstances,CodeStateID,X-Compilable,CompileMessageType,CompileMessageData
0,143,Student1,Assign10,task1.py,Run.Program,NaN,NaN,NaN,NaN,Start,1636762771481,PC;PP 1.1.10,NaN,1,TypeError,"Traceback (most recent call last):\nFile ""task..."
1,144,Student1,Assign10,task1.py,Run.Program,NaN,NaN,NaN,NaN,1,1636762781920,PC;PP 1.1.10,NaN,1,NaN,NaN
2,145,Student1,Assign10,task1.py,Run.Program,NaN,NaN,NaN,NaN,Start,1636762993558,PC;PP 1.1.10,NaN,1,AttributeError,"Traceback (most recent call last):\nFile ""task..."
3,146,Student1,Assign10,task1.py,Run.Program,NaN,NaN,NaN,NaN,1,1636763000044,PC;PP 1.1.10,NaN,1,NaN,NaN
4,147,Student1,Assign10,task1.py,Run.Program,NaN,NaN,NaN,NaN,Start,1636763160150,PC;PP 1.1.10,NaN,1,<NA>,"Traceback (most recent call last):\nFile ""task..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54556,2088849,Student9,Assign9,test.py,Run.Program,NaN,NaN,NaN,NaN,0,1636155782630,PC;PP 1.1.10,NaN,1,NaN,NaN
54557,2088861,Student9,Assign9,test.py,Run.Program,NaN,NaN,NaN,NaN,Start,1636155790509,PC;PP 1.1.10,NaN,1,<NA>,<NA>
54558,2088862,Student9,Assign9,test.py,Run.Program,NaN,NaN,NaN,NaN,0,1636155790573,PC;PP 1.1.10,NaN,1,NaN,NaN
54559,2088864,Student9,Assign9,test.py,Run.Program,NaN,NaN,NaN,NaN,Start,1636156035803,PC;PP 1.1.10,NaN,1,<NA>,<NA>


In [16]:
error_events["ParentEventID"] = pd.NA
error_events.loc[error_events.CompileMessageData.notna(), "ParentEventID"] = \
    error_events.loc[error_events.CompileMessageData.notna()].EventID.astype(str)

In [17]:
uncompilable_runs_without_errors_flag = \
    (error_events[Cols.EventType] == "Run.Program") & \
    (error_events["X-Compilable"] == 0) & \
    error_events[Cols.CompileMessageData].isna()

error_events.loc[uncompilable_runs_without_errors_flag, "CompileMessageType"] = "SyntaxError"

In [18]:
error_events = error_events[~pd.isna(error_events.CompileMessageData)].copy()
error_events["EventID"] = error_events["EventID"].astype(str) + "_error"
error_events[Cols.EventType] = EventType.CompileError.value

In [19]:
keystrokes[Cols.CompileMessageData] = pd.NA
keystrokes[Cols.CompileMessageType] = pd.NA
keystrokes[Cols.ParentEventID] = pd.NA
keystrokes[Cols.EventID] = keystrokes[Cols.EventID].astype(str)

In [20]:
events_with_errors = pd.concat([keystrokes, error_events], ignore_index=True)

In [21]:
events_with_errors[events_with_errors.CompileMessageData.notna()].head()

,EventID,SubjectID,AssignmentID,CodeStateSection,EventType,SourceLocation,EditType,InsertText,DeleteText,X-Metadata,ClientTimestamp,ToolInstances,CodeStateID,X-Compilable,CompileMessageData,CompileMessageType,ParentEventID
2088866,143_error,Student1,Assign10,task1.py,Compile.Error,NaN,NaN,NaN,NaN,Start,1636762771481,PC;PP 1.1.10,NaN,1,"Traceback (most recent call last):\nFile ""task...",TypeError,143
2088867,145_error,Student1,Assign10,task1.py,Compile.Error,NaN,NaN,NaN,NaN,Start,1636762993558,PC;PP 1.1.10,NaN,1,"Traceback (most recent call last):\nFile ""task...",AttributeError,145
2088868,147_error,Student1,Assign10,task1.py,Compile.Error,NaN,NaN,NaN,NaN,Start,1636763160150,PC;PP 1.1.10,NaN,1,"Traceback (most recent call last):\nFile ""task...",<NA>,147
2088869,149_error,Student1,Assign10,task1.py,Compile.Error,NaN,NaN,NaN,NaN,Start,1636763189120,PC;PP 1.1.10,NaN,1,"Traceback (most recent call last):\nFile ""task...",<NA>,149
2088870,151_error,Student1,Assign10,task1.py,Compile.Error,NaN,NaN,NaN,NaN,Start,1636763297345,PC;PP 1.1.10,NaN,1,"Traceback (most recent call last):\nFile ""task...",<NA>,151


In [22]:
grades = pd.read_csv(os.path.join(base_dir, 'students.csv'))
grades.drop(columns=['Unnamed: 0'], inplace=True)
link_subject_path = os.path.join(base_dir, '..', '..', '2021', 'LinkTables', 'Subject.csv')
# create dir if not exists
os.makedirs(os.path.dirname(link_subject_path), exist_ok=True)
grades.to_csv(link_subject_path, index=False)

grades.head()

,SubjectID,Assign6,Assign7,Assign8,Assign9,Assign10,Assign11,Assign12,Assign13,Exam1,Exam2,Exam3,FinalScore,major,HighestACT,HighSchoolGPA
0,Student11,70.0,88.0,45.0,63.0,86.0,97.0,20.0,88.0,82.00,82.0,78.0,89.58,Non-Matriculated,NaN,NaN
1,Student23,100.0,100.0,100.0,98.0,0.0,91.0,94.0,0.0,74.00,78.0,82.0,90.98,Physics,28.0,4.000
2,Student21,90.0,96.0,98.0,95.0,77.0,0.0,97.0,56.0,68.67,78.0,80.0,90.79,Non-Matriculated,NaN,NaN
3,Student17,89.0,96.0,100.0,94.0,91.0,100.0,89.0,0.0,88.00,84.0,80.0,96.95,Statistics,26.0,3.933
4,Student7,100.0,100.0,99.5,97.0,100.0,95.0,98.0,18.0,90.00,90.0,86.0,100.52,Math/Stats - Comp Teaching,35.0,4.000


In [23]:
assignment_cols = [f"Assign{i}" for i in range(6, 13)]
assignment_grades = grades[[Cols.SubjectID] + assignment_cols].copy()
assignment_grades = assignment_grades.melt(id_vars=Cols.SubjectID, var_name=Cols.AssignmentID, value_name=Cols.Score)
assignment_grades

,SubjectID,AssignmentID,Score
0,Student11,Assign6,70.0
1,Student23,Assign6,100.0
2,Student21,Assign6,90.0
3,Student17,Assign6,89.0
4,Student7,Assign6,100.0
...,...,...,...
303,Student26,Assign12,93.0
304,Student44,Assign12,15.0
305,Student14,Assign12,NaN
306,Student4,Assign12,NaN


In [24]:
import numpy as np

last_actions = keystrokes.loc[keystrokes.groupby([Cols.SubjectID, Cols.AssignmentID])[Cols.ClientTimestamp].idxmax()].copy()
last_actions["EventID"] = last_actions[Cols.EventID].astype(str) + "_submit"
last_actions[Cols.EventType] = EventType.Submit.value
last_actions[Cols.ClientTimestamp] = last_actions[Cols.ClientTimestamp] + 1
other_columns = [c for c in last_actions.columns if c not in [Cols.SubjectID, Cols.AssignmentID, Cols.ClientTimestamp, Cols.EventType, Cols.EventID, Cols.ToolInstances]]
last_actions.loc[:, other_columns] = np.nan
last_actions

C:\Users\twprice\AppData\Local\Temp\ipykernel_19012\1516662302.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  last_actions.loc[:, other_columns] = np.nan


,EventID,SubjectID,AssignmentID,CodeStateSection,EventType,SourceLocation,EditType,InsertText,DeleteText,X-Metadata,ClientTimestamp,ToolInstances,CodeStateID,X-Compilable,CompileMessageData,CompileMessageType,ParentEventID
276,276_submit,Student1,Assign10,NaN,Submit,NaN,NaN,NaN,NaN,NaN,1636774253392,PC;PP 1.1.10,NaN,NaN,NaN,NaN,NaN
3318,3318_submit,Student1,Assign12,NaN,Submit,NaN,NaN,NaN,NaN,NaN,1638590392548,PC;PP 1.1.10,NaN,NaN,NaN,NaN,NaN
6568,6568_submit,Student1,Assign13,NaN,Submit,NaN,NaN,NaN,NaN,NaN,1639185447295,PC;PP 1.1.10,NaN,NaN,NaN,NaN,NaN
13945,13945_submit,Student1,Assign6,NaN,Submit,NaN,NaN,NaN,NaN,NaN,1634622810844,PC;PP 1.1.10,NaN,NaN,NaN,NaN,NaN
16169,16169_submit,Student1,Assign7,NaN,Submit,NaN,NaN,NaN,NaN,NaN,1634881571558,PC;PP 1.1.10,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2013217,2013217_submit,Student9,Assign13,NaN,Submit,NaN,NaN,NaN,NaN,NaN,1639126664975,PC;PP 1.1.10,NaN,NaN,NaN,NaN,NaN
2023319,2023319_submit,Student9,Assign6,NaN,Submit,NaN,NaN,NaN,NaN,NaN,1634622629926,PC;PP 1.1.10,NaN,NaN,NaN,NaN,NaN
2051327,2051327_submit,Student9,Assign7,NaN,Submit,NaN,NaN,NaN,NaN,NaN,1634959412406,PC;PP 1.1.10,NaN,NaN,NaN,NaN,NaN
2061301,2061301_submit,Student9,Assign8,NaN,Submit,NaN,NaN,NaN,NaN,NaN,1635552985389,PC;PP 1.1.10,NaN,NaN,NaN,NaN,NaN


In [25]:
submit_actions = last_actions.merge(assignment_grades, on=[Cols.SubjectID, Cols.AssignmentID], how='inner')
submit_actions

,EventID,SubjectID,AssignmentID,CodeStateSection,EventType,SourceLocation,EditType,InsertText,DeleteText,X-Metadata,ClientTimestamp,ToolInstances,CodeStateID,X-Compilable,CompileMessageData,CompileMessageType,ParentEventID,Score
0,276_submit,Student1,Assign10,NaN,Submit,NaN,NaN,NaN,NaN,NaN,1636774253392,PC;PP 1.1.10,NaN,NaN,NaN,NaN,NaN,98.0
1,3318_submit,Student1,Assign12,NaN,Submit,NaN,NaN,NaN,NaN,NaN,1638590392548,PC;PP 1.1.10,NaN,NaN,NaN,NaN,NaN,82.0
2,13945_submit,Student1,Assign6,NaN,Submit,NaN,NaN,NaN,NaN,NaN,1634622810844,PC;PP 1.1.10,NaN,NaN,NaN,NaN,NaN,99.0
3,16169_submit,Student1,Assign7,NaN,Submit,NaN,NaN,NaN,NaN,NaN,1634881571558,PC;PP 1.1.10,NaN,NaN,NaN,NaN,NaN,99.0
4,23203_submit,Student1,Assign8,NaN,Submit,NaN,NaN,NaN,NaN,NaN,1635556758665,PC;PP 1.1.10,NaN,NaN,NaN,NaN,NaN,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
234,2013096_submit,Student9,Assign12,NaN,Submit,NaN,NaN,NaN,NaN,NaN,1638588785255,PC;PP 1.1.10,NaN,NaN,NaN,NaN,NaN,79.0
235,2023319_submit,Student9,Assign6,NaN,Submit,NaN,NaN,NaN,NaN,NaN,1634622629926,PC;PP 1.1.10,NaN,NaN,NaN,NaN,NaN,47.0
236,2051327_submit,Student9,Assign7,NaN,Submit,NaN,NaN,NaN,NaN,NaN,1634959412406,PC;PP 1.1.10,NaN,NaN,NaN,NaN,NaN,76.0
237,2061301_submit,Student9,Assign8,NaN,Submit,NaN,NaN,NaN,NaN,NaN,1635552985389,PC;PP 1.1.10,NaN,NaN,NaN,NaN,NaN,78.0


In [26]:
events_with_errors_and_submits = pd.concat([events_with_errors, submit_actions], ignore_index=True)

In [27]:
events_with_errors_and_submits.to_csv(os.path.join(base_dir, '..', '..', '2021', 'MainTable.csv'), index=False)

In [28]:
error_events[error_events.CompileMessageData.notna() & (error_events["X-Compilable"] == 0)][Cols.CompileMessageType].value_counts()

CompileMessageType
TypeError                     245
SyntaxError                   187
AttributeError                 85
IndexError                     81
IndentationError               69
NameError                      69
ValueError                     42
UnboundLocalError              18
turtle.TurtleGraphicsError     12
ZeroDivisionError               6
Name: count, dtype: int64

In [29]:
error_events[error_events.EventType == "Run.Program"]["X-Compilable"].value_counts()

Series([], Name: count, dtype: int64)

In [30]:
runs.shape

(24194, 14)

In [31]:
keystroke_runs = keystrokes[keystrokes.EventType == 'Run.Program']
keystroke_runs[keystroke_runs[[Cols.SubjectID, Cols.AssignmentID, Cols.ClientTimestamp]].duplicated()]

,EventID,SubjectID,AssignmentID,CodeStateSection,EventType,SourceLocation,EditType,InsertText,DeleteText,X-Metadata,ClientTimestamp,ToolInstances,CodeStateID,X-Compilable,CompileMessageData,CompileMessageType,ParentEventID
106491,106491,Student12,Assign12,task1.py,Run.Program,NaN,NaN,NaN,NaN,0,1638390895118,PC;PP 1.1.10,NaN,1,<NA>,<NA>,<NA>
106501,106501,Student12,Assign12,task1.py,Run.Program,NaN,NaN,NaN,NaN,0,1638390918828,PC;PP 1.1.10,NaN,1,<NA>,<NA>,<NA>
106545,106545,Student12,Assign12,task1.py,Run.Program,NaN,NaN,NaN,NaN,0,1638390953203,PC;PP 1.1.10,NaN,1,<NA>,<NA>,<NA>
106718,106718,Student12,Assign12,task1.py,Run.Program,NaN,NaN,NaN,NaN,0,1638391508127,PC;PP 1.1.10,NaN,1,<NA>,<NA>,<NA>
106722,106722,Student12,Assign12,task1.py,Run.Program,NaN,NaN,NaN,NaN,0,1638391609434,PC;PP 1.1.10,NaN,1,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2084708,2084708,Student9,Assign9,task2.py,Run.Program,NaN,NaN,NaN,NaN,1,1636173437259,PC;PP 1.1.10,NaN,0,<NA>,<NA>,<NA>
2085537,2085537,Student9,Assign9,task2.py,Run.Program,NaN,NaN,NaN,NaN,1,1636175980178,PC;PP 1.1.10,NaN,0,<NA>,<NA>,<NA>
2087730,2087730,Student9,Assign9,test.py,Run.Program,NaN,NaN,NaN,NaN,1,1636130304256,PC;PP 1.1.10,NaN,0,<NA>,<NA>,<NA>
2087881,2087881,Student9,Assign9,test.py,Run.Program,NaN,NaN,NaN,NaN,1,1636151122565,PC;PP 1.1.10,NaN,0,<NA>,<NA>,<NA>


In [32]:
keystroke_runs["X-Compilable"].value_counts()

X-Compilable
1    47959
0     6602
Name: count, dtype: int64

In [33]:
(~runs.CompileMessageData.isna()).sum()

np.int64(8771)

In [34]:
keystroke_runs["X-Metadata"].value_counts()

X-Metadata
Start          27326
0              13230
1               8603
-1              2732
130             1675
-1073741510      653
-805306369       154
137              139
143               31
2                 17
1073807364         1
Name: count, dtype: int64

In [35]:
keystroke_runs[(keystroke_runs.EventID - 106491).abs() < 10]

TypeError: unsupported operand type(s) for -: 'str' and 'int'

In [ ]:
keystrokes[keystrokes.EventType == 'Run.Program'][[Cols.SubjectID, Cols.AssignmentID, Cols.ClientTimestamp]].duplicated().mean()

np.float64(0.05014570847308517)

In [ ]:
runs[[Cols.SubjectID, Cols.AssignmentID, Cols.ClientTimestamp, Cols.CompileMessageType, Cols.CompileMessageData]]

,SubjectID,AssignmentID,ClientTimestamp,CompileMessageType,CompileMessageData
0,Student1,Assign6,1.634537e+12,<NA>,<NA>
1,Student1,Assign6,1.634622e+12,<NA>,<NA>
2,Student1,Assign6,1.634537e+12,<NA>,<NA>
3,Student1,Assign6,1.634622e+12,<NA>,<NA>
4,Student1,Assign6,1.634537e+12,<NA>,<NA>
...,...,...,...,...,...
11574,Student29,Assign13,1.639095e+12,ModuleNotFoundError,"Traceback (most recent call last):\nFile ""memo..."
11575,Student29,Assign13,1.639095e+12,ModuleNotFoundError,"Traceback (most recent call last):\nFile ""memo..."
11576,Student29,Assign13,1.639095e+12,ModuleNotFoundError,"Traceback (most recent call last):\nFile ""memo..."
11577,Student29,Assign13,1.639095e+12,AttributeError,"Traceback (most recent call last):\nFile ""memo..."


In [ ]:
runs[~pd.isna(runs.last_line)].last_line

11                                       KeyboardInterrupt
12                                       KeyboardInterrupt
13                                       KeyboardInterrupt
14                                       KeyboardInterrupt
15                                       KeyboardInterrupt
                               ...                        
11574    ModuleNotFoundError: No module named 'memorycard'
11575    ModuleNotFoundError: No module named 'memorycard'
11576    ModuleNotFoundError: No module named 'memorycard'
11577    AttributeError: 'MemoryBoard' object has no at...
11578    AttributeError: 'MemoryBoard' object has no at...
Name: last_line, Length: 11568, dtype: object

In [ ]:
n = 0
for i, row in runs[~pd.isna(runs.stderr)].iterrows():
    print(row.last_line)
    # print(row.stderr)
    print('---')
    if n > 30:
        break
    n += 1


KeyboardInterrupt
---
KeyboardInterrupt
---
KeyboardInterrupt
---
KeyboardInterrupt
---
TypeError: cannot unpack non-iterable int object
---
TypeError: 'str' object cannot be interpreted as an integer
---
TypeError: 'str' object cannot be interpreted as an integer
---
TypeError: 'str' object is not callable
---
TypeError: can only concatenate str (not "int") to str
---
TypeError: 'str' object is not callable
---
KeyboardInterrupt
---
TypeError: object of type 'int' has no len()
---
KeyboardInterrupt
---
KeyboardInterrupt
---
KeyboardInterrupt
---
KeyboardInterrupt
---
KeyboardInterrupt
---
KeyboardInterrupt
---
turtle.Terminator
---
KeyboardInterrupt
---
turtle.Terminator
---
KeyboardInterrupt
---
KeyboardInterrupt
---
KeyboardInterrupt
---
KeyboardInterrupt
---
KeyboardInterrupt
---
KeyboardInterrupt
---
KeyboardInterrupt
---
KeyboardInterrupt
---
KeyboardInterrupt
---
KeyboardInterrupt
---
KeyboardInterrupt
---


In [ ]:
keystrokes[keystrokes.EventType == 'Run.Program'].head()

,EventID,SubjectID,AssignmentID,CodeStateSection,EventType,SourceLocation,EditType,InsertText,DeleteText,X-Metadata,ClientTimestamp,ToolInstances,CodeStateID,X-Compilable
143,143,Student1,Assign10,task1.py,Run.Program,NaN,NaN,NaN,NaN,Start,1636762771481,PC;PP 1.1.10,NaN,1
144,144,Student1,Assign10,task1.py,Run.Program,NaN,NaN,NaN,NaN,1,1636762781920,PC;PP 1.1.10,NaN,1
145,145,Student1,Assign10,task1.py,Run.Program,NaN,NaN,NaN,NaN,Start,1636762993558,PC;PP 1.1.10,NaN,1
146,146,Student1,Assign10,task1.py,Run.Program,NaN,NaN,NaN,NaN,1,1636763000044,PC;PP 1.1.10,NaN,1
147,147,Student1,Assign10,task1.py,Run.Program,NaN,NaN,NaN,NaN,Start,1636763160150,PC;PP 1.1.10,NaN,1


In [ ]:
runs1[runs1.Source == "PythonRunner"].Output.isna().mean()

np.float64(1.0)

In [ ]:
(runs1.OutputDestination == 'stderr').mean()

np.float64(0.00195840831490462)

In [ ]:
runs1.NumOutputLines.value_counts()

NumOutputLines
1.0        851
0.0        572
10.0       491
5.0        446
8.0        417
          ... 
302.0        1
41347.0      1
93869.0      1
31248.0      1
19999.0      1
Name: count, Length: 334, dtype: int64

In [ ]:
runs1[~runs1.Output.isna()].Output.str.count('\n').mean()

np.float64(0.9967637136885001)

In [ ]:
runs1.NumOutputLines[~runs1.Output.isna()].value_counts()

Series([], Name: count, dtype: int64)

In [ ]:
output = runs1[runs1.OutputDestination == "stderr"].Output.reset_index()
for i in range(100):
    print(output.values[i][1])
    print('-' * 80)

Traceback (most recent call last):

--------------------------------------------------------------------------------
  File "task2.py", line 42, in <module>

--------------------------------------------------------------------------------
    userInput = input("Would you like to run the simulation again? (yes/no): ").lower()

--------------------------------------------------------------------------------
KeyboardInterrupt

--------------------------------------------------------------------------------
Traceback (most recent call last):

--------------------------------------------------------------------------------
  File "flukytester.py", line 11, in <module>

--------------------------------------------------------------------------------
    number = int(input("Enter desired number to check: "))

--------------------------------------------------------------------------------
KeyboardInterrupt

--------------------------------------------------------------------------------
Trace

In [ ]:
def reconstruct(df):
    s = ''
    for _,row in df[df.EventType=='File.Edit'].iterrows():
        i = int(row.SourceLocation)
        insert = '' if pd.isna(row.InsertText) else row.InsertText
        delete = '' if pd.isna(row.DeleteText) else row.DeleteText
        s = s[:i] + insert + s[i+len(delete):]
    return s

In [ ]:
def reconstruct_all(df, in_place=False):
    if not in_place:
        df = df.copy()
    codestates = []
    s = ''
    for _,row in df.iterrows():
        if row.EventType == 'File.Edit':
            i = int(row.SourceLocation)
            insert = '' if pd.isna(row.InsertText) else row.InsertText
            delete = '' if pd.isna(row.DeleteText) else row.DeleteText
            s = s[:i] + insert + s[i+len(delete):]
        codestates.append(s)
    df['CodeState'] = codestates
    return df

In [ ]:
one_attempt = keystrokes[(keystrokes.SubjectID == "S000") & (keystrokes.AssignmentID == "p4f")]
one_attempt

,EventID,SubjectID,AssignmentID,CodeStateSection,X-Task,EventType,X-Keystroke,InsertText,DeleteText,SourceLocation,ClientTimestamp,EditType,X-RunInput,X-RunOutput,X-RunHasError,X-RunUserTerminated,X-RawAssignmentID,X-Term,X-Compilable
0,0,S000,p4f,task0.py,0,File.Edit,#,#,NaN,0.0,1568242704669,Insert,NaN,NaN,NaN,NaN,p4,f,1
1,1,S000,p4f,task0.py,0,File.Edit,space,,NaN,1.0,1568242704830,Insert,NaN,NaN,NaN,NaN,p4,f,1
2,2,S000,p4f,task0.py,0,File.Edit,delete,NaN,,1.0,1568242705198,Delete,NaN,NaN,NaN,NaN,p4,f,1
3,3,S000,p4f,task0.py,0,File.Edit,space,,NaN,1.0,1568242705450,Insert,NaN,NaN,NaN,NaN,p4,f,1
4,4,S000,p4f,task0.py,0,File.Edit,@,@,NaN,2.0,1568242705692,Insert,NaN,NaN,NaN,NaN,p4,f,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1896,1896,S000,p4f,task0.py,0,File.Edit,t,t,NaN,865.0,1568244374034,Insert,NaN,NaN,NaN,NaN,p4,f,0
1897,1897,S000,p4f,task0.py,0,File.Edit,space,,NaN,866.0,1568244374086,Insert,NaN,NaN,NaN,NaN,p4,f,0
1898,1898,S000,p4f,task0.py,0,File.Edit,i,i,NaN,867.0,1568244374182,Insert,NaN,NaN,NaN,NaN,p4,f,0
1899,1899,S000,p4f,task0.py,0,File.Edit,t,t,NaN,868.0,1568244374297,Insert,NaN,NaN,NaN,NaN,p4,f,0


In [ ]:
with_codestate = reconstruct_all(one_attempt)

with_codestate

,EventID,SubjectID,AssignmentID,CodeStateSection,X-Task,EventType,X-Keystroke,InsertText,DeleteText,SourceLocation,ClientTimestamp,EditType,X-RunInput,X-RunOutput,X-RunHasError,X-RunUserTerminated,X-RawAssignmentID,X-Term,X-Compilable,CodeState
0,0,S000,p4f,task0.py,0,File.Edit,#,#,NaN,0.0,1568242704669,Insert,NaN,NaN,NaN,NaN,p4,f,1,#
1,1,S000,p4f,task0.py,0,File.Edit,space,,NaN,1.0,1568242704830,Insert,NaN,NaN,NaN,NaN,p4,f,1,#
2,2,S000,p4f,task0.py,0,File.Edit,delete,NaN,,1.0,1568242705198,Delete,NaN,NaN,NaN,NaN,p4,f,1,#
3,3,S000,p4f,task0.py,0,File.Edit,space,,NaN,1.0,1568242705450,Insert,NaN,NaN,NaN,NaN,p4,f,1,#
4,4,S000,p4f,task0.py,0,File.Edit,@,@,NaN,2.0,1568242705692,Insert,NaN,NaN,NaN,NaN,p4,f,1,# @
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1896,1896,S000,p4f,task0.py,0,File.Edit,t,t,NaN,865.0,1568244374034,Insert,NaN,NaN,NaN,NaN,p4,f,0,# @@@@@@@@@@@@@@@\n# But you can call me @@@@\...
1897,1897,S000,p4f,task0.py,0,File.Edit,space,,NaN,866.0,1568244374086,Insert,NaN,NaN,NaN,NaN,p4,f,0,# @@@@@@@@@@@@@@@\n# But you can call me @@@@\...
1898,1898,S000,p4f,task0.py,0,File.Edit,i,i,NaN,867.0,1568244374182,Insert,NaN,NaN,NaN,NaN,p4,f,0,# @@@@@@@@@@@@@@@\n# But you can call me @@@@\...
1899,1899,S000,p4f,task0.py,0,File.Edit,t,t,NaN,868.0,1568244374297,Insert,NaN,NaN,NaN,NaN,p4,f,0,# @@@@@@@@@@@@@@@\n# But you can call me @@@@\...


In [ ]:
for i in range(len(with_codestate)):
    if i % 100 == 0:
        print(with_codestate.iloc[i].CodeState)
        print('\n-------\n')

#

-------

# @@@@@@@@@@@@@@@
# Assn # 4
#creates a snowman that technically fil

-------

# @@@@@@@@@@@@@@@
# Assn # 4
#creates a snowman that technically fills all the points in the rubric
#his name is legalese-man
import turtle,
def create_circle()

-------

# @@@@@@@@@@@@@@@
# Assn # 4
#creates a snowman that technically fills all the points in the rubric
#his name is legalese-man
import turtle
def createCircl

-------

# @@@@@@@@@@@@@@@
# Assn # 4
#creates a snowman that technically meets all the points in the rubric
#his name is legalese-man
import turtle

#create a function crea
def createCircle

-------

# @@@@@@@@@@@@@@@
# Assn # 4
#creates a snowman that technically meets all the points in the rubric
#his name is legalese-man
import turtle

#create a function creates a circle.  Takes multiple sizes, and can be filled or not.  Also a color 
def createCircle(Size, col):

-------

# @@@@@@@@@@@@@@@
# Assn # 4
#creates a snowman that technically meets all the points in the rubric

In [ ]:
from spec.enums import MainTableColumns as Cols
from spec.spec_definition import PS2Versions
import pandas as pd

In [ ]:
from database.config import PS2DataConfig

spec = PS2Versions.v1_0.load()

data_config = PS2DataConfig.from_yaml("sample_data_configs/codebench2024.yaml", spec)
